# CAP VMT Analysis (Loop)

In [49]:
import os
import pandas as pd
import math
from IPython.display import display
import numpy as py
import ast
import warnings
# warnings.filterwarnings(action='once')
warnings.filterwarnings('ignore')
import openpyxl
import time
import datetime
import pyodbc

import inro.modeller as _m
import inro.emme.database.emmebank as _eb

## User Defined Inputs

### Choose District Boundary
#### Jurisdiction (JUR) or Community Planning Area (CPA)

In [50]:
# enter "JUR" or "CPA"
district_boundary = "JUR"

#### Define selected zone to run select link analysis

In [ ]:
# if not needing to run select link analysis, enter False to skip the step; the script will run data summary only
run_select_link = True

# coose select zone id number
select_zone_id_loop_list = [range(1, 5), range(5, 9), range(9, 13), range(13, 17), range(17, 20)]
select_zone_id_loop_list

[range(1, 5), range(5, 9), range(9, 13), range(13, 17), range(17, 20)]

#### Do you need to delete existing select link results from EMME database (network volumn and demand matrices)?

In [52]:
# enter True only if you want to delete the results from database. They CANNOT be recovered. 
delete_option = True
initial_query_to_delete = ["carl", "chul", "coro", "delm",
                           "elca", "enci", "esco", "impe", 
                           "lame", "lemo", "nati", "ocea", 
                           "powa", "sand", "sanm", "sant", 
                           "sola", "vist", "unin"
                          ]

#### Do you need to expand the size of database? (Default to False)

In [53]:
# enter True only if the scenario has more than 4 select link analysis done. 
# Note the database expansion process could take more than 2 hours

expand_option = True

# based on number of selected regions in query set this number. 40000000 is recommended for 4 regions
required_dimensions = 40000000

# set required matrices if including many select link queries. Model default is 1600, each SL will add 75 tables in ABM2+
# required_matrices = 2500

## Initialization

In [54]:
# program timer initiated
start_time_program = time.time()

In [55]:
modeller = _m.Modeller()
desktop = modeller.desktop
emmebank = modeller.emmebank

project_dir = os.path.dirname(_m.Modeller().desktop.project.path)
main_dir = os.path.dirname(project_dir)
database_dir = os.path.join(project_dir, "Database")
cap_vmt_dir = os.path.join(main_dir, "analysis", "cap_vmt")
cap_vmt_input_dir = os.path.join(cap_vmt_dir, "input")
report_dir = os.path.join(main_dir, "report")
output_dir = os.path.join(cap_vmt_dir, "output")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [56]:
# EMME tools
change_db_dimension = modeller.tool("inro.emme.data.database.change_database_dimensions")
import_attr_values = modeller.tool("inro.emme.data.network.import_attribute_values")
create_attr = modeller.tool("inro.emme.data.extra_attribute.create_extra_attribute")
copy_attr = modeller.tool("inro.emme.data.network.copy_attribute")
traffic_assign  = modeller.tool("sandag.assignment.traffic_assignment")
delete_matrix = modeller.tool("inro.emme.data.matrix.delete_matrix")

In [57]:
# EMME time-of-day scenarios
scenario_id = 100
periods = ["EA", "AM", "MD", "PM", "EV"]
period_ids = list(enumerate(periods, start=int(scenario_id) + 1))

In [58]:
# emmebank and networks
main_emmebank = _eb.Emmebank(os.path.join(project_dir, "Database", "emmebank"))
base_scenario = emmebank.scenario(scenario_id)

base_network = base_scenario.get_network()

zones = emmebank.scenario(scenario_id).zone_numbers

In [59]:
# report template path
template_path = os.path.join(cap_vmt_input_dir,"vmt_CAP_Template.xlsx").replace("\\","/")
template = openpyxl.load_workbook(template_path)

In [60]:
# post-porcessing highway link - cpa lookup file to include links outside cpa 
hwy_jur = pd.read_csv(os.path.join(cap_vmt_input_dir,"hwy_jur.csv"), sep=',')
hwy_cpa = pd.read_csv(os.path.join(cap_vmt_input_dir,"hwy_cpa.csv"), sep=',')
colnames = ["tcov_id", "dist", "length"]
hwy_jur.columns = colnames
hwy_cpa.columns = colnames

hwy_out_cpa = hwy_jur[(hwy_jur.dist != 14) & (hwy_jur.dist != 19)]
hwy_cpa = hwy_cpa.append(hwy_out_cpa)
        
hwy_cpa.to_csv(os.path.join(cap_vmt_input_dir, "hwy_cpa_all.csv"), header = True, index= False)

In [61]:
# input lookup files
if district_boundary.lower() == "jur":
    xfer_taz_jur_path = os.path.join(cap_vmt_input_dir,"xref_taz15_jur.csv").replace("\\","/")
    xref_taz_jur_unique_path = os.path.join(cap_vmt_input_dir,"xref_taz15_jur_unique.csv").replace("\\","/")
    xref_link_jur_path = os.path.join(cap_vmt_input_dir,"hwy_jur.csv").replace("\\","/")
elif district_boundary.lower() == "cpa":
    xfer_taz_jur_path = os.path.join(cap_vmt_input_dir,"xref_taz15_cpa.csv").replace("\\","/")
    xref_taz_jur_unique_path = os.path.join(cap_vmt_input_dir,"xref_taz15_cpa_unique.csv").replace("\\","/")
    xref_link_jur_path = os.path.join(cap_vmt_input_dir,"hwy_cpa_all.csv").replace("\\","/")
else: 
    raise ValueError("Please enter correct district boundary choice.")

zone_correspondence = pd.read_csv(xfer_taz_jur_path,sep=',')
xref_taz_jur_unique = pd.read_csv(xref_taz_jur_unique_path,sep=',')

In [62]:
# post processing link-district coorespondence files to calculate split factors for links crossing multiple districts
link_correspondence = pd.read_csv(xref_link_jur_path, sep=',')
colnames = ["tcov_id", "dist", "length"]
link_correspondence.columns = colnames

length_sum = link_correspondence.groupby(["tcov_id"])["length"].sum()
length_sum_df = length_sum.to_frame()
length_sum_df.reset_index(inplace=True)
length_sum_df.columns = ['tcov_id', "total_length"]   

link_correspondence = link_correspondence.merge(length_sum_df, on="tcov_id", how="left")
link_correspondence["split_factor"] = link_correspondence.length / link_correspondence.total_length
link_correspondence = link_correspondence[['tcov_id', "dist", "split_factor"]]

In [63]:
# read in fc look up file
fc_lookup_path = os.path.join(cap_vmt_input_dir,"fc_lookup.csv").replace("\\","/")
fc_lookup = pd.read_csv(fc_lookup_path, sep=',')
xref_taz_jur_unique = pd.read_csv(xref_taz_jur_unique_path,sep=',')

In [64]:
# vehicle classes
auto_classes = [
    'SOV_NT_L', 'SOV_TR_L', 'HOV2_L', 'HOV3_L',
    'SOV_NT_M', 'SOV_TR_M', 'HOV2_M', 'HOV3_M',
    'SOV_NT_H', 'SOV_TR_H', 'HOV2_H', 'HOV3_H'
]
truck_classes = [
    'TRK_L', 'TRK_M', 'TRK_H'
]
truck_classes_pce = [
    ('TRK_L', 1.3),
    ('TRK_M', 1.5),
    ('TRK_H', 2.5)
]

In [65]:
# define speed bin for VMT summary by speed
speed_bin = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 999]

## Helper Functions

In [66]:
# populate select link query based on specified zone id
def get_select_link_query(select_zone_id):
    select_link_query = []
    suffix_list = []
    for id in select_zone_id:
        suffix = xref_taz_jur_unique[xref_taz_jur_unique.group_id == id]['Suffix'].tolist()[0]
        suffix_list.append(suffix)
        query_string = '{"expression": "@selected_zone = %i or @selected_zonej = %i", "suffix": "%s", "threshold": 1}' % (id, id, suffix)
        query = ast.literal_eval(query_string)
        select_link_query.append(query)
    return select_link_query, suffix_list

In [67]:
# function to delete extra attributes for specified select link queries on network
def delete_extra_attribute(suffixes):
    for suffix in suffixes:
        suffix = str(suffix)
        auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]

        truck_attrs = ["@sl_" + name.lower() + "_" + suffix
                       for name in truck_classes]

        for number, period, in enumerate(periods, start=scenario_id + 1):
            scenario = main_emmebank.scenario(number)    
            for name in auto_attrs: 
                if scenario.extra_attribute(name):
                    scenario.delete_extra_attribute(name)            
            for name in truck_attrs: 
                if scenario.extra_attribute(name):
                    scenario.delete_extra_attribute(name)

            if scenario.extra_attribute("@slink_"+ suffix):
                scenario.delete_extra_attribute("@slink_"+ suffix) 

In [68]:
# function to delete demand matrices for specified select link queries in database
def delete_matrices(suffixes):
    matrix_id_list = []
    matrix_id_name = []
    for suffix in suffixes: 
        suffix = str(suffix)
        for matrix in main_emmebank.matrices():
            if suffix in matrix.description  :
                matrix_id_list.append(matrix.id)

    for i in range(0, (len(matrix_id_list))):
            delete_matrix(matrix_id_list[i])

In [69]:
# function to expand EMME database dimension
def expand_database_dimension(required_dimensions):
    new_dimensions = emmebank.dimensions
    if new_dimensions["extra_attribute_values"] < required_dimensions:
        new_dimensions["extra_attribute_values"] = required_dimensions
        change_db_dimension(emmebank_dimensions=new_dimensions,
                  keep_backup=True)

In [70]:
# function to create extra attributes on EMME networks
def create_extra_attributes():
    scenario = main_emmebank.scenario(101)
    create_attr("NODE", "@selected_zone", "jurisdiction", overwrite=True, scenario=scenario)
    import_attr_values(xfer_taz_jur_path,
                  field_separator = ",",
                  scenario = scenario,
                  column_labels={0: "inode", 
                                 1: "@selected_zone"},
                  revert_on_error=True)

    for scenario_id in range(102, 106):
        time_scenario = main_emmebank.scenario(scenario_id)
        create_attr("NODE", "@selected_zone", "jurisdiction", overwrite=True, scenario=time_scenario)
        copy_attr(from_attribute_name = "@selected_zone",
                  to_attribute_name = "@selected_zone", 
                  from_scenario = scenario, 
                  to_scenario = time_scenario)    

In [71]:
# select link analysis
def run_select_link_analysis(select_link_query):
    msa_iteration = 4
    relative_gap = 0.0005
    max_assign_iterations = 100
    num_processors = "MAX-1"
    select_link = select_link_query

    for number, period in period_ids:
        period_scenario = main_emmebank.scenario(number)
        traffic_assign(period, msa_iteration, relative_gap, max_assign_iterations,num_processors, period_scenario, select_link)

In [72]:
def get_ref_network():
    base_network = base_scenario.get_network()
    ref_networks = []
    for number, period, in enumerate(periods, start=scenario_id + 1):
        scenario = emmebank.scenario(number)
        network = scenario.get_network()
        base_scenario.publish_network(base_network)
        ref_networks.append(scenario.get_network())  
    return ref_networks

In [73]:
def summarize_select_link(suffix, base_scenario, network, ref_networks):
    suffix = str(suffix)
    total_name = "@sel_" + suffix
    if base_scenario.extra_attribute(total_name):
        base_scenario.delete_extra_attribute(total_name)
    flow_xatt = base_scenario.create_extra_attribute("LINK", total_name)
    flow_xatt.description = "Total select flow for %s" % suffix
    if total_name in network.attributes("LINK"):
        network.delete_attribute("LINK", total_name)
    network.create_attribute("LINK", total_name) 
    
    total_flow = "@total_flow"
    if base_scenario.extra_attribute(total_flow):
        base_scenario.delete_extra_attribute(total_flow)
    total_flow_xatt = base_scenario.create_extra_attribute("LINK", total_flow)
    total_flow_xatt.description = "Total daily link flow"
    if total_flow in network.attributes("LINK"):
        network.delete_attribute("LINK", total_flow)
    network.create_attribute("LINK", total_flow)
    
    auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]
    truck_attrs = [("@sl_" + name.lower() + "_" + suffix, pce)
                   for name, pce in truck_classes_pce]

    for ref_network in ref_networks:
        for link in base_network.links():
            ref_link = ref_network.link(link.i_node, link.j_node)
            flow = sum(ref_link[att] for att in auto_attrs)
            flow += sum(ref_link[att]/pce for att, pce in truck_attrs)
            link[total_name] += flow
            link[total_flow] += ref_link["@non_pce_flow"]
  
    base_scenario.publish_network(base_network)

In [74]:
def export_selec_link_load(zone_id, suffix, base_network):
    id_format = lambda x: str(int(x))
    total_names = "@sel_" + str(suffix)
    hwyload_attrs = [("tcov_id", "@tcov_id"),("length","length"), ("FC", "type")]
    dir_atts = [("AB_flow", "@total_flow"), ("AB_sel", total_names), ("AB_lanes", "num_lanes")]
    for key, attr in dir_atts:
        hwyload_attrs.append((key, attr))
        hwyload_attrs.append((key.replace("AB_", "BA_"), (attr, "")))  # default for BA on one-way links is blank

    auto_mode = base_network.mode("d")
    # only the original forward direction links and auto links only
    links = [l for l in base_network.links() 
             if l["@tcov_id"] > 0 and 
             (auto_mode in l.modes or (l.reverse_link and auto_mode in l.reverse_link.modes))
            ]
    links.sort(key=lambda l: l["@tcov_id"])

    id_suffix = str(zone_id) + "_" + suffix
    suffix_dir = os.path.join(output_dir, id_suffix)
    loadselk_file = os.path.join(suffix_dir, ("loadselk_" + id_suffix + ".csv"))
    with open(loadselk_file, 'w') as fout:
        fout.write(",".join(['"%s"' % x[0] for x in hwyload_attrs]))
        fout.write("\n")
        for link in links:
            key, att = hwyload_attrs[0]  # expected to be the link id
            values = [id_format(link[att])]
            reverse_link = link.reverse_link
            for key, att in hwyload_attrs[1:]:
                if key.startswith("BA"):
                    name, default = att
                    if reverse_link and (abs(link["@tcov_id"]) == abs(reverse_link["@tcov_id"])):
                        values.append(format(reverse_link[name]))
                    else:
                        values.append(default)
                    #values.append(format(reverse_link[name]) if reverse_link else default)
                elif att.startswith("#"):
                    values.append('"%s"' % link[att])
                else:
                    values.append(format(link[att]))
            fout.write(",".join(values))
            fout.write("\n")

    loadselk = pd.read_csv(loadselk_file,sep=',')
    loadselk = loadselk.fillna(0)
    loadselk = loadselk.merge(link_correspondence, on="tcov_id", how="left")
    names=[]
    AB_sel_Flow = 'AB_sel'
    BA_sel_Flow = 'BA_sel'
    total_sel_flow = "Total_Flow_sel"
    AB_sel_VMT = "AB_VMT_sel"
    BA_sel_VMT = "BA_VMT_sel"
    total_sel_VMT = "Total_VMT_sel"    
    loadselk[AB_sel_Flow] = loadselk[AB_sel_Flow]
    loadselk[BA_sel_Flow] = loadselk[BA_sel_Flow]
    loadselk[total_sel_flow] = loadselk[AB_sel_Flow] + loadselk[BA_sel_Flow] 
    loadselk[AB_sel_VMT] = loadselk[AB_sel_Flow ] * loadselk['length'] 
    loadselk[BA_sel_VMT] = loadselk[BA_sel_Flow] * loadselk['length'] 
    loadselk[total_sel_VMT] = loadselk[total_sel_flow] * loadselk['length']
          
    AB_Flow = 'AB_flow'
    BA_Flow = 'BA_flow'
    total_flow = "Total_Flow"
    AB_VMT = "AB_VMT"
    BA_VMT = "BA_VMT"
    total_VMT = "Total_VMT"
    loadselk[AB_Flow] = loadselk[AB_Flow]
    loadselk[BA_Flow] = loadselk[BA_Flow]
    loadselk[total_flow] = loadselk[AB_Flow] + loadselk[BA_Flow] 
    loadselk[AB_VMT] = loadselk[AB_Flow ] * loadselk['length'] 
    loadselk[BA_VMT] = loadselk[BA_Flow] * loadselk['length'] 
    loadselk[total_VMT] = loadselk[total_flow] * loadselk['length']
    
    
    loadselk['length'] = loadselk['length'] * loadselk["split_factor"]    
    loadselk[AB_sel_Flow] = loadselk[AB_sel_Flow] * loadselk["split_factor"]
    loadselk[BA_sel_Flow] = loadselk[BA_sel_Flow] * loadselk["split_factor"]
    loadselk[total_sel_flow] = loadselk[AB_sel_Flow] + loadselk[BA_sel_Flow] 
    loadselk[AB_sel_VMT] = loadselk[AB_sel_VMT ] * loadselk['split_factor'] 
    loadselk[BA_sel_VMT] = loadselk[BA_sel_VMT] * loadselk['split_factor'] 
    loadselk[total_sel_VMT] = loadselk[AB_sel_VMT] + loadselk[BA_sel_VMT]
    
    loadselk[AB_Flow] = loadselk[AB_Flow] * loadselk["split_factor"]
    loadselk[BA_Flow] = loadselk[BA_Flow] * loadselk["split_factor"]
    loadselk[total_flow] = loadselk[AB_Flow] + loadselk[BA_Flow] 
    loadselk[AB_VMT] = loadselk[AB_VMT ] * loadselk['split_factor'] 
    loadselk[BA_VMT] = loadselk[BA_VMT] * loadselk['split_factor'] 
    loadselk[total_VMT] = loadselk[AB_VMT] + loadselk[BA_VMT]
    loadselk["lanemile"] = (loadselk["AB_lanes"] + loadselk["BA_lanes"]) * loadselk['length']
    
    names.extend([AB_Flow,BA_Flow,total_flow,AB_VMT,BA_VMT,total_VMT,
                  AB_sel_Flow,BA_sel_Flow,total_sel_flow,AB_sel_VMT,BA_sel_VMT,total_sel_VMT]) 
    colnames = ["tcov_id", "dist", "length", "FC", "lanemile", "AB_lanes", "BA_lanes"] + names
    loadselk = loadselk[colnames]
    
    loadselk.to_csv(loadselk_file, index= False)
    
    return loadselk

In [75]:
def total_vmt_summary(zone_id, loadselk, ii_ratio, intra_vmt):
    loadselk_summary = (loadselk.groupby(['dist']))["Total_VMT", "Total_VMT_sel"].sum()
    loadselk_summary.reset_index(inplace=True)
    loadselk_summary.columns = ['Dist', "Total_VMT", "Total_VMT_sel"]
    loadselk_summary["sel_vmt_II"] = 0
    loadselk_summary.loc[loadselk_summary['Dist'] == zone_id, "sel_vmt_II"] = loadselk_summary.Total_VMT_sel * ii_ratio
    loadselk_summary["sel_vmt_IE_EI"] = loadselk_summary.Total_VMT_sel - loadselk_summary.sel_vmt_II
    loadselk_summary["sel_vmt_EE"] = loadselk_summary.Total_VMT - loadselk_summary.Total_VMT_sel
    loadselk_summary["intra_vmt"] = 0
    loadselk_summary.loc[loadselk_summary['Dist'] == zone_id, "intra_vmt"] = intra_vmt
    loadselk_summary = loadselk_summary.merge(xref_taz_jur_unique, left_on="Dist", right_on="group_id", how="left")
    # remove districts with null values
    loadselk_summary = loadselk_summary[loadselk_summary['Name'].notnull()]
    loadselk_summary = loadselk_summary.append(loadselk_summary.sum(numeric_only=True), ignore_index=True)
    colnames = ["Name", "Total_VMT", "Total_VMT_sel", "sel_vmt_II", "sel_vmt_IE_EI", "sel_vmt_EE", "intra_vmt"]
    loadselk_summary = loadselk_summary[colnames]
    loadselk_summary = loadselk_summary.sort_values(by = ['Name'], ascending = [True], na_position = 'last')
    loadselk_summary.iat[-1, 0] = "Total"   
    
    return loadselk_summary

In [76]:
def vmt_fc(zone_id, loadselk, ii_ratio, intra_vmt):

    loadselk_fc = (loadselk.groupby(['dist', 'FC']))["Total_VMT", "Total_VMT_sel"].sum()
    loadselk_fc.reset_index(inplace=True)
    loadselk_fc.columns = ['Dist', 'FC', "Total_VMT", "Total_VMT_sel"] 
    loadselk_fc["sel_vmt_II"] = 0
    loadselk_fc.loc[loadselk_fc['Dist'] == zone_id, "sel_vmt_II"] = loadselk_fc.Total_VMT_sel * ii_ratio
    loadselk_fc["sel_vmt_IE_EI"] = loadselk_fc.Total_VMT_sel - loadselk_fc.sel_vmt_II
    loadselk_fc["sel_vmt_EE"] = loadselk_fc.Total_VMT - loadselk_fc.Total_VMT_sel
    loadselk_fc = loadselk_fc.merge(xref_taz_jur_unique, left_on="Dist", right_on="group_id", how="left")
    loadselk_fc =  loadselk_fc.merge(fc_lookup, on="FC", how="right")
    loadselk_fc["intra_vmt"] = 0
    colnames = ["Name", 'FC', "Class", "Total_VMT", "Total_VMT_sel", "sel_vmt_II", "sel_vmt_IE_EI", "sel_vmt_EE", "intra_vmt"]
    loadselk_fc = loadselk_fc[colnames]

    # remove districts with null values
    loadselk_fc = loadselk_fc[loadselk_fc['Name'].notnull()]

    # calculate grand total
    cols = ["Total_VMT", "Total_VMT_sel", "sel_vmt_II", "sel_vmt_IE_EI", "sel_vmt_EE", "intra_vmt"]
    grand = loadselk_fc[cols].sum()
    grand.loc['Name'] = 'Grand total'

    study_area = get_study_area_name(zone_id)

    # calculate subtotal for each district outside of study area
    fc_out =  loadselk_fc[loadselk_fc.Name != study_area]
    fc_out_1 = fc_out.groupby('Name')[cols].sum()
    fc_out_1.index = fc_out_1.index + ' Sub Total'
    fc_out = pd.concat([fc_out.set_index('Name'), fc_out_1], keys=('a', 'b')).sort_index(level=1).reset_index()
    fc_out = fc_out.drop('level_0', axis=1)


    # within study area, to add intra zonal vmt to the last row
    fc_in = loadselk_fc[loadselk_fc.Name == study_area]
    fc_in.loc[-1] = [study_area, 11, "Intra Zonal", 0, 0 ,0, 0, 0, intra_vmt]

    fc_in_1 = fc_in.groupby('Name')[cols].sum()
    fc_in_1.index = fc_in_1.index + ' Sub Total'
    fc_in = pd.concat([fc_in.set_index('Name'), fc_in_1], keys=('a', 'b')).sort_index(level=1).reset_index()
    fc_in = fc_in.drop('level_0', axis=1)

    # combine outside and within study area
    fc_all = fc_in.append(fc_out)
    fc_all = fc_all.sort_values(by = ['Name', 'FC'], ascending = [True, True], na_position = 'first')
    #add grand total
    fc_all.loc[len(fc_all.index)] = grand

    fc_all = fc_all[colnames]
    fc_all.iat[-1, -1] = intra_vmt 
    fc_all = fc_all.replace(py.nan, '', regex=True)
    
    return fc_all

In [77]:
def summary_vmt_by_speed_bin(zone_id, suffix, speed_bin):
    auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]
    truck_attrs = [("@sl_" + name.lower() + "_" + suffix, pce)
                   for name, pce in truck_classes_pce]
    
    # get link VMT summary in scenario 101
    first_period = periods[0]
    scenario = main_emmebank.scenario(scenario_id + 1)
    network = scenario.get_network()
    auto_mode = network.mode("d")

    tcov_id_list = []
    length_list = []
    speed_list = []
    sl_flow_list = []
    for link in network.links():
        if auto_mode in link.modes:
            tcov_id = link["@tcov_id"]
            tcov_id_list.append(tcov_id)
            length = link["length"]
            length_list.append(length)
            speed = link["@speed"]
            speed_list.append(speed)
            sl_flow = sum(link[att] for att in auto_attrs)
            sl_flow += sum(link[att] / pce for att, pce in truck_attrs)
            sl_flow_list.append(sl_flow)
    list_of_fields = list(zip(tcov_id_list, length_list, speed_list, sl_flow_list))
    
    speed_var = "speed_" + first_period
    sl_flow_var = "sl_flow_" + first_period
    vmt_var = "vmt_" + first_period
    speed_df_1 = pd.DataFrame(list_of_fields, columns = ['tcov_id', 'length', speed_var, sl_flow_var])
    # turning tcov_id to positive and join with link_jur lookup to remove null districts
    speed_df_1['tcov_id_abs'] = speed_df_1['tcov_id'].abs()
    speed_df_1 = speed_df_1.merge(link_correspondence, left_on='tcov_id_abs', right_on="tcov_id", how="left")
    speed_df_1 = speed_df_1[speed_df_1['dist'] != 0]
    # calculate vmt and apply split factor by length to account for duplicated links in the lookup
    speed_df_1[vmt_var] = speed_df_1.length * speed_df_1[sl_flow_var] * speed_df_1.split_factor
    
    # summarize by speed bin
    speed_summary_1 = (speed_df_1.groupby(pd.cut(speed_df_1[speed_var], bins=speed_bin, right=False)))[vmt_var].sum()
    speed_summary_1_df = speed_summary_1.to_frame()
    speed_summary_1_df.reset_index(inplace=True)
    speed_summary_1_df.columns = ['speed_bin', vmt_var]
    
    # get speed and vmt summary for remaining periods and merge to scenario 101
    remaining_periods = periods[1:]
    for number, period, in enumerate(remaining_periods, start=scenario_id + 2):
        scenario = emmebank.scenario(number)
        network = scenario.get_network()
        auto_mode = network.mode("d")

        tcov_id_list = []
        length_list = []
        speed_list = []
        sl_flow_list = []
        for link in network.links():
            if auto_mode in link.modes:
                tcov_id = link["@tcov_id"]
                tcov_id_list.append(tcov_id)
                length = link["length"]
                length_list.append(length)              
                speed = link["@speed"]
                speed_list.append(speed)    
                sl_flow = sum(link[att] for att in auto_attrs)
                sl_flow += sum(link[att] / pce for att, pce in truck_attrs)
                sl_flow_list.append(sl_flow)        
        list_of_fields = list(zip(tcov_id_list, length_list, speed_list, sl_flow_list))  
        
        speed_var = "speed_" + period
        sl_flow_var = "sl_flow_" + period
        vmt_var = "vmt_" + period
        speed_df = pd.DataFrame(list_of_fields, columns = ['tcov_id', 'length', speed_var, sl_flow_var]) 
        # turning tcov_id to positive and join with link_jur lookup to remove null districts
        speed_df['tcov_id_abs'] = speed_df['tcov_id'].abs()
        speed_df = speed_df.merge(link_correspondence, left_on='tcov_id_abs', right_on="tcov_id", how="left")
        speed_df = speed_df[speed_df['dist'] != 0]
        
        speed_df[vmt_var] = speed_df.length * speed_df[sl_flow_var] * speed_df.split_factor
        
        speed_summary = (speed_df.groupby(pd.cut(speed_df[speed_var], bins=speed_bin, right=False)))[vmt_var].sum()
        speed_summary_df = speed_summary.to_frame()
        speed_summary_df.reset_index(inplace=True)
        speed_summary_df.columns = ['speed_bin', vmt_var]        
        speed_summary_1_df = speed_summary_1_df.merge(speed_summary_df, on="speed_bin" , how="left")
        
        
    id_suffix = str(zone_id) + "_" + suffix    
    suffix_dir = os.path.join(output_dir, id_suffix)

    speed_summary_file_name = "speed_vmt_summary_" + id_suffix  +".csv"
    speed_summary_1_df.to_csv(os.path.join(suffix_dir, speed_summary_file_name), header = True, index= False)
    
    return speed_summary_1_df

In [78]:
def agg_sl_demand(zone_id, suffix):
    suffix = str(suffix)
    agg=[]
    matrix_id_list=[]
    for matrix in emmebank.matrices():
        if matrix.type == "FULL" and "Selected demand" in matrix.description and suffix in matrix.description :
            matrix_id_list.append(matrix.id)

    mat=[]
    for i in matrix_id_list :
        mat.append(emmebank.matrix(id=i).get_numpy_data())
    demand = reduce(lambda x,y: x+y,  mat)
    demand = pd.DataFrame(demand)
    group_I = zone_correspondence[zone_correspondence['Suffix']== suffix ]['zone_id'] 
    all_zones = [zones]
    group_E = set(all_zones[0]) - set(group_I)
    index_I = group_I-1   
    index_E = py.array((map(int,group_E) )) -1
    II=demand.iloc[(index_I)].iloc[:,index_I].values.sum()
    IE=demand.iloc[index_I].iloc[:,index_E].values.sum()
    EI=demand.iloc[index_E].iloc[:,index_I].values.sum()
    EE=demand.iloc[index_E].iloc[:,index_E].values.sum()
    aggregated_demand = pd.DataFrame ({ 'col1' :[EE,IE] , 'col2' : [EI,II] })
    
    id_suffix = str(zone_id) + "_" + suffix
    sl_agg_file_name = "SL_Agg_SLDaily_" + id_suffix +".csv"
    suffix_dir = os.path.join(output_dir, id_suffix)
    sl_agg_file_path = os.path.join(suffix_dir, sl_agg_file_name)
    aggregated_demand.to_csv(sl_agg_file_path, header = False, index= False)
    
    return aggregated_demand

In [79]:
def intrazonal_trip(period) :
    matrix_id_list = []
    matrix_id_name = []
    ##choose selected demand matrices from emmebank and add them to get the total demand
    for matrix in emmebank.matrices():
        if matrix.type == "FULL" and period in matrix.description and "demand" in matrix.description and "Selected" not in matrix.description:
            matrix_id_list.append(matrix.id)

    mat_intra = []
    for i in matrix_id_list :
        mat_intra.append(emmebank.matrix(id=i).get_numpy_data().diagonal())
    trip_intra = reduce(lambda x,y: x+y,  mat_intra)
    intra_trip = pd.DataFrame({period: trip_intra , 'TAZ':zones})
    
    return intra_trip

In [80]:
def intrazonal_summary():
    for matrix in emmebank.matrices():
        if matrix.type == "FULL" and matrix.description == "AM SOV non-transponder medium VOT distance":
            skim = emmebank.matrix(id = matrix.id).get_numpy_data()
            skim_intra = pd.DataFrame({"Intra Distance": skim.diagonal() , 'TAZ':zones}) 

    intra_summary = reduce(lambda x,y: pd.merge(x,y, on="TAZ" , how="left"), 
                  [skim_intra,intrazonal_trip("EA"),intrazonal_trip("AM"),intrazonal_trip("MD"),intrazonal_trip("PM"),intrazonal_trip("EV")])
    intra_summary = intra_summary[['TAZ', "Intra Distance", 'EA', 'AM', 'MD', 'PM', 'EV']]
    intra_summary["Intra_Trips"] = intra_summary[['EA', 'AM', 'MD', 'PM', 'EV']].sum(axis=1)
    intra_summary["intra_vmt"] = intra_summary["Intra Distance"] * intra_summary["Intra_Trips"]
    intra_summary = intra_summary.merge(zone_correspondence, left_on="TAZ", right_on="zone_id", how="left")
    
    intrazonal_out_file_name = "Intrazonal_summary.csv" 
    intrazonal_out_file_path = os.path.join(output_dir, intrazonal_out_file_name)    
    intra_summary.to_csv(intrazonal_out_file_path, index= False)
    
    return intra_summary

In [81]:
def get_study_area_name(zone_id):
    study_area_name = xref_taz_jur_unique[xref_taz_jur_unique.group_id == zone_id]['Name'].tolist()[0]
    return study_area_name

In [82]:
def get_scenario_name():
    path = os.path.split(project_dir)[0]
    scenario_name = os.path.split(path)[1]
    return scenario_name

In [83]:
def vmt_per_lane_mile(zone_id, loadselk, ii_ratio, intra_vmt, total_distance, intra_mile):
    total_vmt = total_vmt_summary(zone_id, loadselk, ii_ratio, intra_vmt)
    name = get_study_area_name(zone_id)
    sel = total_vmt[total_vmt.Name == name]
    sel["total_vmt_sel_permile"] = sel.Total_VMT_sel / total_distance
    sel["sel_vmt_ii_permile"] = sel.sel_vmt_II / total_distance
    sel["sel_vmt_ie_ei_permile"] = sel.sel_vmt_IE_EI / total_distance
    sel["sel_vmt_ee_permile"] = sel.sel_vmt_EE / total_distance
    sel["intra_vmt_permile"] = sel.intra_vmt / intra_mile

    colnames = ["Total_VMT_sel", "total_vmt_sel_permile", "sel_vmt_II", "sel_vmt_ii_permile", "sel_vmt_IE_EI", 
                "sel_vmt_ie_ei_permile", "sel_vmt_EE", "sel_vmt_ee_permile", "intra_vmt", "intra_vmt_permile"]
    sel = sel[colnames]
    
    return sel

In [84]:
# write results to excel spreadsheet

def write_to_excel(zone_id, suffix, loadselk, agg_demand, speed_summary, intra_summary):
    id_suffix = str(zone_id) + "_" + suffix
    suffix_dir = os.path.join(output_dir, id_suffix)
    report_out_file_name = "CAP_VMT_" + id_suffix +".xlsx"    
    report_out_file_path = os.path.join(suffix_dir, report_out_file_name)
    
    templateWriter = pd.ExcelWriter(
        path=report_out_file_path,
        mode="w",
        engine="openpyxl")
    templateWriter.book = template
    templateWriter.sheets = dict((ws.title, ws) for ws in template.worksheets)
   
    speed_summary = speed_summary.iloc[:, 1:]
    speed_summary.to_excel(
        excel_writer=templateWriter,
        sheet_name="Speed Bin Report",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=3,
        startcol=3,
        engine="openpyxl")
    
    agg_demand.to_excel(
        excel_writer=templateWriter,
        sheet_name="Input",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=5,
        startcol=11,
        engine="openpyxl")
    
    ii_ratio = agg_demand.iloc[1,1] / (agg_demand.iloc[0].sum() + agg_demand.iloc[1].sum())
    ii_ratio_df = pd.DataFrame({ii_ratio})
    ii_ratio_df.to_excel(
        excel_writer=templateWriter,
        sheet_name="Input",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=10,
        startcol=15,
        engine="openpyxl")
       
    intra_vmt = intra_summary[intra_summary.group_id == zone_id]["intra_vmt"].sum()
    intra_mile = intra_summary[intra_summary.group_id == zone_id]["Intra Distance"].sum()
    total_lanemile = loadselk[loadselk.dist == zone_id]["lanemile"].sum()
    
    total_lanemile_df = pd.DataFrame({total_lanemile})
    total_lanemile_df.to_excel(
        excel_writer=templateWriter,
        sheet_name="LaneMile",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=3,
        startcol=2,
        engine="openpyxl")
    
    intra = pd.DataFrame([intra_mile, intra_vmt])
    intra.to_excel(
        excel_writer=templateWriter,
        sheet_name="IZ Report",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=3,
        startcol=2,
        engine="openpyxl")
         
    total_vmt = total_vmt_summary(zone_id, loadselk, ii_ratio, intra_vmt)    
    total_vmt.to_excel(
        excel_writer=templateWriter,
        sheet_name="Summary",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=6,
        startcol=1,
        engine="openpyxl")
    
    fc_vmt = vmt_fc(zone_id, loadselk, ii_ratio, intra_vmt)
    fc_vmt.to_excel(
        excel_writer=templateWriter,
        sheet_name="Analysis",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=8,
        startcol=1,
        engine="openpyxl")  
      
    vmt_lane_mile = vmt_per_lane_mile(zone_id, loadselk, ii_ratio, intra_vmt, total_lanemile, intra_mile)
    vmt_lane_mile.to_excel(
        excel_writer=templateWriter,
        sheet_name="LaneMile",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=3,
        startcol=3,
        engine="openpyxl")
    
    name = pd.DataFrame({get_study_area_name(zone_id)})
    name.to_excel(
        excel_writer=templateWriter,
        sheet_name="Input",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=6,
        startcol=3,
        engine="openpyxl") 
    
    regional_vmt_python = total_vmt.iloc[-1, 1]
    regional_vmt_python_df = pd.DataFrame({regional_vmt_python})
    regional_vmt_python_df.to_excel(
        excel_writer=templateWriter,
        sheet_name="Input",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=12,
        startcol=3,
        engine="openpyxl") 
    
    name = get_study_area_name(zone_id)
    clip_vmt = total_vmt.loc[total_vmt['Name'] == name, "Total_VMT"]
    clip_vmt_df = pd.DataFrame({clip_vmt.tolist()[0]})
    clip_vmt_df.to_excel(
        excel_writer=templateWriter,
        sheet_name="Input",
        na_rep="NULL",
        header=False,
        index=False,
        startrow=13,
        startcol=3,
        engine="openpyxl") 
 
    # save the completed template
    templateWriter.save()

## CAP VMT Procedure

####  (Optional) Delete Redundant Extra Attributes and Demand Matrices

In [85]:
if delete_option == True:
    delete_extra_attribute(initial_query_to_delete)
    delete_matrices(initial_query_to_delete)

#### (Optional) Expand Database Dimension

In [86]:
if expand_option == True:
    expand_database_dimension(required_dimensions)

In [87]:
print "Finished Database Expansion in %5.2f mins" % ((time.time() - start_time_program)/60.0)

Finished Database Expansion in  0.11 mins


#### Run Select Link Analysis (run time is about 30 min for each query)

In [88]:
create_extra_attributes()
intra_summary = intrazonal_summary()

In [89]:
for select_zone_id in select_zone_id_loop_list:
    sl_start_time = time.time()
    select_link_query, suffix_list = get_select_link_query(select_zone_id)
    
    # create subfolders for each query
    id_suffix_list = zip(select_zone_id, suffix_list)
    for zone_id, suffix in id_suffix_list:
        suffix_sub_dir = os.path.join(output_dir, (str(zone_id) + "_" + suffix))
        if not os.path.exists(suffix_sub_dir):
            os.makedirs(suffix_sub_dir)
            
    # run select link analysis
    run_select_link_analysis(select_link_query)
    
    # generate select link summary for each query
    ref_networks = get_ref_network()
    for zone_id, suffix in id_suffix_list:
        summarize_select_link(suffix, base_scenario, base_network, ref_networks)
        loadselk = export_selec_link_load(zone_id, suffix, base_network)
        speed_summary = summary_vmt_by_speed_bin(zone_id, suffix, speed_bin)
        agg_demand = agg_sl_demand(zone_id, suffix)
        write_to_excel(zone_id, suffix, loadselk, agg_demand, speed_summary, intra_summary)
    
    # delete select link results from network and matrix to free up space
    delete_extra_attribute(suffix_list)
    delete_matrices(suffix_list)
    print "Select link analysis done for zones %s in %5.2f mins" %(suffix_list, ((time.time() - sl_start_time)/60.0))

Select link analysis done for zones ['elca', 'enci', 'esco', 'impe'] in 139.71 mins
Select link analysis done for zones ['lame', 'lemo', 'nati', 'ocea'] in 141.57 mins
Select link analysis done for zones ['powa', 'sanm', 'sant', 'sola'] in 140.62 mins
Select link analysis done for zones ['vist', 'unin'] in 98.76 mins


#### Generate Select Link Summary

In [90]:
print "Finished Select Link Analysis and Summary in %5.2f mins" % ((time.time() - start_time_program)/60.0)

Finished Select Link Analysis and Summary in 522.34 mins
